In [ ]:
# @title A. Config: fixed n=1024 table run, S-RAS K=768, pointwise K=256

import os, gc, math, itertools, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as stats
from tqdm.notebook import tqdm

import jax
import jax.numpy as jnp
from jax import random

# =========================================================
# Table design
# =========================================================

N_TABLE = 1024
DATA_SPLIT_TABLE = "test"
IMAGE_SUBSET_SEED = 20260505

K_MAIN = [16, 32, 64, 128]
K_SRAS_REF = 768
K_POINTWISE_REF = 256

K_SRAS_ALL = K_MAIN + [K_SRAS_REF]
K_POINTWISE_ALL = K_MAIN + [K_POINTWISE_REF]

EPS_REG = 1e-4
RANDOM_PARENT_SEED = 0

BASIS_MODE_TABLE = "both"  # "random", "pca", or "both"

# JVP chunk for geo_engine.get_grams_per_sample.
# If JVP OOMs, reduce to 4 or 2.
GEOM_CHUNK_SRAS = 4
GEOM_CHUNK_POINTWISE = 8

# Activation extraction chunk.
ACT_CACHE_CHUNK = 64

RUN_TAG = (
    f"tiny10_layer_table_n{N_TABLE}_"
    f"srasK{K_SRAS_REF}_pwK{K_POINTWISE_REF}_"
    f"{BASIS_MODE_TABLE}_{DATA_SPLIT_TABLE}_seed{IMAGE_SUBSET_SEED}"
)

DRIVE_OUTDIR = Path("/content/drive/MyDrive/layer_matching_table_data") / RUN_TAG
DRIVE_OUTDIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = DRIVE_OUTDIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", DRIVE_OUTDIR)
print("Cache directory:", CACHE_DIR)
print("JAX backend:", jax.default_backend())
print("Local device count:", jax.local_device_count())
print("N_TABLE:", N_TABLE)
print("S-RAS K values:", K_SRAS_ALL)
print("Pointwise K values:", K_POINTWISE_ALL)

PAIR_LIST = list(itertools.combinations(range(len(loaded_models)), 2))
print("Model pairs:", len(PAIR_LIST))

# =========================================================
# Robust sharding helper
# =========================================================

def shard_for_pmap_exact(x):
    n = x.shape[0]
    num_devices = jax.local_device_count()
    if n % num_devices != 0:
        raise ValueError(f"N={n} is not divisible by local_device_count={num_devices}.")
    return x.reshape((num_devices, n // num_devices) + x.shape[1:])

# =========================================================
# CIFAR-10 loading without tfds
# =========================================================

def _as_numpy_image_array(x):
    x = np.asarray(x)

    if x.ndim != 4 or x.shape[-1] != 3:
        raise ValueError(f"Expected image array [N,H,W,3], got shape {x.shape}")

    if x.dtype != np.float32:
        x = x.astype(np.float32)

    if x.max() > 2.0:
        x = (x - 127.5) / 127.5

    return x.astype(np.float32)

def _load_keras_cifar10_split(split):
    print("Loading full CIFAR-10 via keras.datasets.cifar10...")
    try:
        from tensorflow.keras.datasets import cifar10
    except Exception:
        from keras.datasets import cifar10

    (x_train, _), (x_test, _) = cifar10.load_data()

    split = str(split).lower()
    if split in ["test", "validation", "val"]:
        arr = _as_numpy_image_array(x_test)
    elif split in ["train", "training"]:
        arr = _as_numpy_image_array(x_train)
    else:
        raise ValueError(f"Unknown split={split!r}; use 'train' or 'test'.")

    print("Loaded Keras CIFAR-10 split:", split, arr.shape, arr.dtype)
    return arr

print(f"Loading CIFAR-10 split={DATA_SPLIT_TABLE!r} without tfds...")
X_all = _load_keras_cifar10_split(DATA_SPLIT_TABLE)

if X_all.shape[0] < N_TABLE:
    raise ValueError(
        f"Requested N_TABLE={N_TABLE}, but split has only {X_all.shape[0]} images."
    )

rng = np.random.default_rng(IMAGE_SUBSET_SEED)
image_indices = rng.choice(np.arange(X_all.shape[0]), size=N_TABLE, replace=False)
image_indices = np.asarray(image_indices, dtype=np.int64)

X_table = X_all[image_indices].astype(np.float32)
X_table_sharded = shard_for_pmap_exact(X_table)

pd.DataFrame({
    "subset_position": np.arange(N_TABLE),
    "cifar10_index": image_indices,
}).to_csv(DRIVE_OUTDIR / f"fixed_{N_TABLE}_image_indices.csv", index=False)

print("X_table:", X_table.shape, "->", X_table_sharded.shape)
print("Saved image indices:", DRIVE_OUTDIR / f"fixed_{N_TABLE}_image_indices.csv")

In [ ]:
# @title B. Build random/PCA parent bases for S-RAS K=768 and pointwise K=256

flat_dim = int(np.prod(X_table.shape[1:]))

families_sras = {}
families_pointwise = {}

def make_random_parent(flat_dim, k, seed):
    key = random.PRNGKey(seed)
    P = random.normal(key, (flat_dim, k), dtype=jnp.float32)
    P, _ = jnp.linalg.qr(P)
    return np.asarray(P[:, :k], dtype=np.float32)

if BASIS_MODE_TABLE in ["random", "both"]:
    P_RANDOM_SRAS = make_random_parent(flat_dim, K_SRAS_REF, RANDOM_PARENT_SEED)
    P_RANDOM_PW = P_RANDOM_SRAS[:, :K_POINTWISE_REF].astype(np.float32)

    families_sras["random"] = P_RANDOM_SRAS
    families_pointwise["random"] = P_RANDOM_PW

    print("Random S-RAS parent:", P_RANDOM_SRAS.shape)
    print("Random pointwise parent:", P_RANDOM_PW.shape)

if BASIS_MODE_TABLE in ["pca", "both"]:
    X_flat = X_table.reshape(N_TABLE, -1).astype(np.float32)
    X_flat_centered = X_flat - X_flat.mean(axis=0, keepdims=True)

    U, S, Vt = np.linalg.svd(X_flat_centered, full_matrices=False)
    rank_available = Vt.shape[0]

    if K_SRAS_REF > rank_available:
        raise ValueError(
            f"K_SRAS_REF={K_SRAS_REF} exceeds PCA rank {rank_available}. "
            f"Use more images or lower K_SRAS_REF."
        )

    P_PCA_SRAS = Vt[:K_SRAS_REF].T.astype(np.float32)
    P_PCA_PW = P_PCA_SRAS[:, :K_POINTWISE_REF].astype(np.float32)

    explained_var = (S ** 2) / max(X_flat_centered.shape[0] - 1, 1)
    explained_ratio = explained_var / np.maximum(explained_var.sum(), 1e-12)
    cumulative = np.cumsum(explained_ratio)

    pca_meta = pd.DataFrame({
        "pc_index_1_based": np.arange(1, len(S) + 1),
        "singular_value": S,
        "explained_variance": explained_var,
        "explained_variance_ratio": explained_ratio,
        "cumulative_explained_variance_ratio": cumulative,
    })
    pca_meta.to_csv(DRIVE_OUTDIR / f"pca_family_metadata_fixed{N_TABLE}.csv", index=False)

    families_sras["whitened_pca"] = P_PCA_SRAS
    families_pointwise["whitened_pca"] = P_PCA_PW

    print("PCA S-RAS parent:", P_PCA_SRAS.shape)
    print("PCA pointwise parent:", P_PCA_PW.shape)

    for k in K_SRAS_ALL:
        print(f"PCA cumulative variance at K={k}: {cumulative[k-1]:.4f}")

print("S-RAS families:", list(families_sras.keys()))
print("Pointwise families:", list(families_pointwise.keys()))

In [ ]:

# @title C. Activation baselines on fixed n=1024 benchmark

def _center_rows(X):
    return X - X.mean(axis=0, keepdims=True)

def _linear_gram(Xc):
    return (Xc @ Xc.T).astype(np.float32)

def _double_center_gram(K):
    K = np.asarray(K, dtype=np.float32)
    return K - K.mean(axis=0, keepdims=True) - K.mean(axis=1, keepdims=True) + K.mean()

def _pairwise_sq_dists(X):
    X = np.asarray(X, dtype=np.float32)
    sq = np.sum(X * X, axis=1, keepdims=True)
    D2 = sq + sq.T - 2.0 * (X @ X.T)
    return np.maximum(D2, 0.0).astype(np.float32)

def _rbf_gram_median_heuristic(Xc):
    D2 = _pairwise_sq_dists(Xc)
    tri = D2[np.triu_indices(D2.shape[0], k=1)]
    tri = tri[tri > 0]
    sigma2 = float(np.median(tri)) if tri.size else 1.0
    sigma2 = max(sigma2, 1e-12)
    return _double_center_gram(np.exp(-D2 / (2.0 * sigma2)).astype(np.float32))

def _cka_from_grams(K, L):
    num = float(np.trace(K @ L))
    den = float(np.linalg.norm(K) * np.linalg.norm(L)) + 1e-12
    return num / den

def _left_svd_from_Xc(Xc, floor=1e-12):
    G = (Xc @ Xc.T).astype(np.float64)
    G = 0.5 * (G + G.T)
    evals, U = np.linalg.eigh(G)
    keep = evals > floor
    evals = evals[keep]
    U = U[:, keep]
    s = np.sqrt(np.maximum(evals, floor))
    order = np.argsort(s)[::-1]
    return U[:, order].astype(np.float32), s[order].astype(np.float32)

def _ensure_activation_entry(entry, do_rbf=True):
    if "U" not in entry or "s" not in entry:
        entry["U"], entry["s"] = _left_svd_from_Xc(entry["Xc"])
    if "K_linear" not in entry:
        entry["K_linear"] = _linear_gram(entry["Xc"])
    if do_rbf and "K_rbf" not in entry:
        entry["K_rbf"] = _rbf_gram_median_heuristic(entry["Xc"])

def _procrustes_similarity(entry_x, entry_y):
    _ensure_activation_entry(entry_x, do_rbf=False)
    _ensure_activation_entry(entry_y, do_rbf=False)
    Ux, sx = entry_x["U"], entry_x["s"]
    Uy, sy = entry_y["U"], entry_y["s"]
    core = (sx[:, None] * (Ux.T @ Uy).astype(np.float32)) * sy[None, :]
    nuc = np.linalg.svd(core, compute_uv=False, full_matrices=False).sum()
    den = float(np.linalg.norm(sx) * np.linalg.norm(sy)) + 1e-12
    return float(nuc / den)

def _cca_corrs(entry_x, entry_y):
    _ensure_activation_entry(entry_x, do_rbf=False)
    _ensure_activation_entry(entry_y, do_rbf=False)
    corrs = np.linalg.svd(entry_x["U"].T @ entry_y["U"], compute_uv=False, full_matrices=False)
    return np.clip(corrs.astype(np.float32), 0.0, 1.0)

def _cca_r2(entry_x, entry_y):
    corrs = _cca_corrs(entry_x, entry_y)
    return float(np.mean(corrs ** 2))

def _pair_accuracy_from_sim_map(sim_map):
    n = sim_map.shape[0]
    hits = 0
    for i in range(n):
        hits += int(np.argmax(sim_map[i, :]) == i)
    for j in range(n):
        hits += int(np.argmax(sim_map[:, j]) == j)
    return 100.0 * hits / (2 * n)

def _sem(x):
    x = np.asarray(x, dtype=np.float64)
    return float(stats.sem(x)) if len(x) > 1 else 0.0

def _make_summary_row(method, family, regime, K, n_images, pair_accs, note):
    pair_accs = np.asarray(pair_accs, dtype=np.float64)
    return {
        "method": method,
        "family": family,
        "regime": regime,
        "K": "" if K is None else int(K),
        "n_images": int(n_images),
        "n_pairs": int(len(pair_accs)),
        "pair_accuracy_mean_pct": float(pair_accs.mean()),
        "pair_accuracy_sem_pct": _sem(pair_accs),
        "overall_accuracy_pct": float(pair_accs.mean()),
        "note": note,
    }

def _make_pair_rows(method, family, regime, K, pair_accs):
    rows = []
    for (m1, m2), acc in zip(PAIR_LIST, pair_accs):
        rows.append({
            "method": method,
            "family": family,
            "regime": regime,
            "K": "" if K is None else int(K),
            "model_i": int(m1),
            "model_j": int(m2),
            "pair_accuracy_pct": float(acc),
        })
    return rows

def _evaluate_similarity_pair_accs(item_lookup, metric_fn, desc):
    n_layers = len(LAYER_NAMES)
    pair_accs = []

    for idx1, idx2 in tqdm(PAIR_LIST, desc=desc, leave=False):
        sim_map = np.empty((n_layers, n_layers), dtype=np.float32)
        for i, l1 in enumerate(LAYER_NAMES):
            e1 = item_lookup[idx1][l1]
            for j, l2 in enumerate(LAYER_NAMES):
                e2 = item_lookup[idx2][l2]
                sim_map[i, j] = metric_fn(e1, e2)
        pair_accs.append(_pair_accuracy_from_sim_map(sim_map))

    return np.asarray(pair_accs, dtype=np.float32)

activation_summary_path = DRIVE_OUTDIR / f"activation_fixed{N_TABLE}_summary.csv"
activation_pair_path = DRIVE_OUTDIR / f"activation_fixed{N_TABLE}_pair_accuracies.csv"

if activation_summary_path.exists() and activation_pair_path.exists():
    print("Activation results already exist. Loading.")
    activation_summary_df = pd.read_csv(activation_summary_path)
    activation_pair_df = pd.read_csv(activation_pair_path)
else:
    activation_cache = {}

    for m in tqdm(range(len(loaded_models)), desc=f"Activation cache n={N_TABLE}"):
        activation_cache[m] = {}
        for layer in LAYER_NAMES:
            X = get_activation_matrix_cpu_chunked(
                loaded_models[m],
                layer,
                X_table_sharded,
                chunk_size=ACT_CACHE_CHUNK,
            )
            Xc = _center_rows(X).astype(np.float32)
            activation_cache[m][layer] = {"Xc": Xc}
        gc.collect()

    print("Preparing activation Grams and SVD factors...")
    for m in tqdm(range(len(loaded_models)), desc="Activation derived caches"):
        for layer in LAYER_NAMES:
            _ensure_activation_entry(activation_cache[m][layer], do_rbf=True)
        gc.collect()

    activation_summary_rows = []
    activation_pair_rows = []

    activation_metrics = {
        "cka_linear": lambda ex, ey: _cka_from_grams(ex["K_linear"], ey["K_linear"]),
        "cka_rbf": lambda ex, ey: _cka_from_grams(ex["K_rbf"], ey["K_rbf"]),
        "procrustes": _procrustes_similarity,
        "cca_r2": _cca_r2,
    }

    for method, fn in activation_metrics.items():
        accs = _evaluate_similarity_pair_accs(
            activation_cache,
            metric_fn=fn,
            desc=f"{method} n={N_TABLE}",
        )
        activation_summary_rows.append(
            _make_summary_row(
                method=method,
                family="activation",
                regime=f"fixed_{N_TABLE}",
                K=None,
                n_images=N_TABLE,
                pair_accs=accs,
                note=f"Activation baseline evaluated on the same fixed random {N_TABLE}-image subset.",
            )
        )
        activation_pair_rows.extend(
            _make_pair_rows(method, "activation", f"fixed_{N_TABLE}", None, accs)
        )

    activation_summary_df = pd.DataFrame(activation_summary_rows)
    activation_pair_df = pd.DataFrame(activation_pair_rows)

    activation_summary_df.to_csv(activation_summary_path, index=False)
    activation_pair_df.to_csv(activation_pair_path, index=False)

display(activation_summary_df)

In [ ]:
# @title D. Compute/cache dataset-level mean operators for S-RAS K=768

SRAS_MEAN_DIR = CACHE_DIR / f"sras_mean_ops_n{N_TABLE}_k{K_SRAS_REF}"
SRAS_MEAN_DIR.mkdir(parents=True, exist_ok=True)

def _safe_layer_name(layer):
    return str(layer).replace("/", "__").replace(" ", "_")

def _sras_mean_path(family, model_i, layer):
    return SRAS_MEAN_DIR / f"{family}_model{model_i:02d}_{_safe_layer_name(layer)}_meanK{K_SRAS_REF}.npy"

def _compute_or_load_sras_mean(family, P_parent, model_i, layer):
    path = _sras_mean_path(family, model_i, layer)
    if path.exists():
        return np.load(path).astype(np.float32)

    grams_ps = geo_engine.get_grams_per_sample(
        loaded_models[model_i],
        layer,
        X_table_sharded,
        k=K_SRAS_REF,
        chunk_size=GEOM_CHUNK_SRAS,
        P_override=P_parent,
    )
    grams_ps = np.asarray(grams_ps, dtype=np.float32)
    mean_G = grams_ps.mean(axis=0).astype(np.float32)

    np.save(path, mean_G)

    del grams_ps
    gc.collect()
    try:
        jax.clear_caches()
    except Exception:
        pass

    return mean_G

# Compute all missing mean operators.
for family, P_parent in families_sras.items():
    print(f"\n=== S-RAS mean operators: {family} ===")
    for m in tqdm(range(len(loaded_models)), desc=f"{family}: models"):
        for layer in tqdm(LAYER_NAMES, desc=f"model {m}", leave=False):
            _ = _compute_or_load_sras_mean(family, P_parent, m, layer)
        gc.collect()

print("Finished S-RAS mean-operator cache:", SRAS_MEAN_DIR)

In [ ]:
# @title E. Compute/cache pointwise Grams for pw-AIRM/MSA K=256

PW_GRAM_DIR = CACHE_DIR / f"pointwise_grams_n{N_TABLE}_k{K_POINTWISE_REF}"
PW_GRAM_DIR.mkdir(parents=True, exist_ok=True)

def _pw_gram_path(family, model_i, layer):
    return PW_GRAM_DIR / f"{family}_model{model_i:02d}_{_safe_layer_name(layer)}_gramsK{K_POINTWISE_REF}.npy"

def _compute_or_load_pointwise_grams(family, P_parent, model_i, layer):
    path = _pw_gram_path(family, model_i, layer)
    if path.exists():
        return path

    grams_ps = geo_engine.get_grams_per_sample(
        loaded_models[model_i],
        layer,
        X_table_sharded,
        k=K_POINTWISE_REF,
        chunk_size=GEOM_CHUNK_POINTWISE,
        P_override=P_parent,
    )
    grams_ps = np.asarray(grams_ps, dtype=np.float32)
    np.save(path, grams_ps)

    del grams_ps
    gc.collect()
    try:
        jax.clear_caches()
    except Exception:
        pass

    return path

for family, P_parent in families_pointwise.items():
    print(f"\n=== Pointwise Grams: {family} ===")
    for m in tqdm(range(len(loaded_models)), desc=f"{family}: models"):
        for layer in tqdm(LAYER_NAMES, desc=f"model {m}", leave=False):
            _ = _compute_or_load_pointwise_grams(family, P_parent, m, layer)
        gc.collect()

print("Finished pointwise Gram cache:", PW_GRAM_DIR)

In [ ]:
# @title F. Evaluate S-RAS, shape S-RAS, pw-AIRM, and MSA from caches

REGIME_NAME = f"fixed_{N_TABLE}"

LOCAL_PAIR_PATH = DRIVE_OUTDIR / f"local_geometry_fixed{N_TABLE}_pair_accuracies.csv"
LOCAL_SUMMARY_PATH = DRIVE_OUTDIR / f"local_geometry_fixed{N_TABLE}_summary.csv"

# =========================================================
# S-RAS helpers
# =========================================================

def _sym(A):
    return 0.5 * (A + A.T)

def _normalize_trace(G, eps=1e-12):
    G = _sym(np.asarray(G, dtype=np.float32))
    tr = float(np.trace(G))
    return G / max(tr, eps)

def _spd_lift_trace_scaled(G, eps_reg=EPS_REG):
    G = _sym(np.asarray(G, dtype=np.float32))
    k = G.shape[0]
    tr = float(np.trace(G))
    scale = tr / k if tr > 0 else 1.0
    return G + eps_reg * scale * np.eye(k, dtype=np.float32)

def _airm(A, B, floor=1e-12):
    A = _sym(np.asarray(A, dtype=np.float64))
    B = _sym(np.asarray(B, dtype=np.float64))

    evals, evecs = np.linalg.eigh(A)
    inv_sqrt = evecs @ np.diag(1.0 / np.sqrt(np.maximum(evals, floor))) @ evecs.T

    mid = _sym(inv_sqrt @ B @ inv_sqrt)
    lam = np.linalg.eigvalsh(mid)
    return float(np.sqrt(np.sum(np.log(np.maximum(lam, floor)) ** 2)))

def _sras_distance(G1, G2, k, shape_only=False):
    G1 = np.asarray(G1[:k, :k], dtype=np.float32)
    G2 = np.asarray(G2[:k, :k], dtype=np.float32)

    if shape_only:
        G1 = _normalize_trace(G1)
        G2 = _normalize_trace(G2)

    A = _spd_lift_trace_scaled(G1, EPS_REG)
    B = _spd_lift_trace_scaled(G2, EPS_REG)
    return _airm(A, B) / np.sqrt(k)

# =========================================================
# JAX pointwise helper
# =========================================================

def _jax_sym_batch(A):
    return 0.5 * (A + jnp.swapaxes(A, -1, -2))

def _jax_spd_lift_trace_scaled_batch(Gs, eps_reg):
    Gs = _jax_sym_batch(Gs)
    k = Gs.shape[-1]
    tr = jnp.trace(Gs, axis1=-2, axis2=-1)[..., None, None]
    scale = jnp.where(tr > 0, tr / k, 1.0)
    return Gs + eps_reg * scale * jnp.eye(k, dtype=Gs.dtype)

@jax.jit
def _pointwise_airm_msa_singleK_batched(A, B):
    k = A.shape[-1]

    A = _jax_spd_lift_trace_scaled_batch(A, EPS_REG)
    B = _jax_spd_lift_trace_scaled_batch(B, EPS_REG)

    evals, evecs = jnp.linalg.eigh(A)
    inv = 1.0 / jnp.sqrt(jnp.maximum(evals, 1e-12))
    A_inv_sqrt = evecs @ (inv[..., :, None] * jnp.swapaxes(evecs, -1, -2))

    mid = _jax_sym_batch(A_inv_sqrt @ B @ A_inv_sqrt)
    lam = jnp.linalg.eigvalsh(mid)
    lam = jnp.maximum(lam, 1e-12)

    d_airm = jnp.sqrt(jnp.sum(jnp.log(lam) ** 2, axis=-1)) / jnp.sqrt(float(k))
    d_airm = jnp.mean(d_airm, axis=1)

    d_msa = 1.0 - jnp.sqrt(lam[..., 0] / lam[..., -1])
    d_msa = jnp.mean(d_msa, axis=1)

    return d_airm, d_msa

# These are aggressive but should match your known-good profile at K=256.
PW_IMAGE_CHUNK_BY_K = {
    16: 256,
    32: 256,
    64: 256,
    128: 256,
    256: 256,
}
LAYERPAIR_BATCH_BY_K = {
    16: 32,
    32: 32,
    64: 32,
    128: 32,
    256: 32,
}

assert N_TABLE % 256 == 0, "N_TABLE must be divisible by 256 for this batching."

def _pointwise_layerpair_distances(GA_path, GB_path, k):
    GA = np.load(GA_path, mmap_mode="r")
    GB = np.load(GB_path, mmap_mode="r")

    img_chunk = PW_IMAGE_CHUNK_BY_K[k]
    # This function handles one layer-pair at a time, so layerpair batch is used outside.
    acc_airm = 0.0
    acc_msa = 0.0
    n_seen = 0

    for s in range(0, N_TABLE, img_chunk):
        e = s + img_chunk
        A_np = np.asarray(GA[s:e, :k, :k], dtype=np.float32)[None, ...]
        B_np = np.asarray(GB[s:e, :k, :k], dtype=np.float32)[None, ...]

        d_airm, d_msa = _pointwise_airm_msa_singleK_batched(
            jnp.asarray(A_np),
            jnp.asarray(B_np),
        )
        d_airm = float(np.asarray(d_airm)[0])
        d_msa = float(np.asarray(d_msa)[0])

        acc_airm += d_airm * (e - s)
        acc_msa += d_msa * (e - s)
        n_seen += (e - s)

    return acc_airm / n_seen, acc_msa / n_seen

def _pointwise_block_distances(family, model_i, model_j, lp_block, k):
    img_chunk = PW_IMAGE_CHUNK_BY_K[k]

    B_lp = len(lp_block)
    accum_airm = np.zeros((B_lp,), dtype=np.float64)
    accum_msa = np.zeros((B_lp,), dtype=np.float64)
    n_seen = 0

    # Open memmaps once per layer-pair block.
    GA_list = []
    GB_list = []
    for a, b in lp_block:
        layer_a = LAYER_NAMES[a]
        layer_b = LAYER_NAMES[b]
        GA_list.append(np.load(_pw_gram_path(family, model_i, layer_a), mmap_mode="r"))
        GB_list.append(np.load(_pw_gram_path(family, model_j, layer_b), mmap_mode="r"))

    for s in range(0, N_TABLE, img_chunk):
        e = s + img_chunk

        A_np = np.stack([
            np.asarray(GA[s:e, :k, :k], dtype=np.float32)
            for GA in GA_list
        ], axis=0)

        B_np = np.stack([
            np.asarray(GB[s:e, :k, :k], dtype=np.float32)
            for GB in GB_list
        ], axis=0)

        d_airm, d_msa = _pointwise_airm_msa_singleK_batched(
            jnp.asarray(A_np),
            jnp.asarray(B_np),
        )

        d_airm = np.asarray(d_airm)
        d_msa = np.asarray(d_msa)

        accum_airm += d_airm * (e - s)
        accum_msa += d_msa * (e - s)
        n_seen += (e - s)

        del A_np, B_np, d_airm, d_msa
        gc.collect()

    return accum_airm / n_seen, accum_msa / n_seen

# =========================================================
# Evaluation/checkpoint helpers
# =========================================================

def _local_pair_complete(pair_df, family, model_i, model_j):
    if len(pair_df) == 0:
        return False

    g = pair_df[
        (pair_df["family"].astype(str) == family) &
        (pair_df["model_i"].astype(int) == int(model_i)) &
        (pair_df["model_j"].astype(int) == int(model_j))
    ]

    needed = (
        2 * len(K_SRAS_ALL) +     # sras_full, sras_shape
        2 * len(K_POINTWISE_ALL)  # pwairm, msa
    )
    return len(g.drop_duplicates(subset=["method", "K"])) >= needed

def _save_local(pair_df):
    pair_df = pair_df.drop_duplicates(
        subset=["method", "family", "regime", "K", "model_i", "model_j"],
        keep="last",
    ).sort_values(["family", "model_i", "model_j", "method", "K"]).reset_index(drop=True)

    pair_df.to_csv(LOCAL_PAIR_PATH, index=False)

    rows = []
    for (method, family, regime, K), g in pair_df.groupby(["method", "family", "regime", "K"], dropna=False):
        vals = g["pair_accuracy_pct"].to_numpy(dtype=np.float64)
        rows.append({
            "method": method,
            "family": family,
            "regime": regime,
            "K": int(K),
            "n_images": int(N_TABLE),
            "n_pairs": int(len(vals)),
            "pair_accuracy_mean_pct": float(vals.mean()),
            "pair_accuracy_sem_pct": float(vals.std(ddof=1) / np.sqrt(len(vals))) if len(vals) > 1 else 0.0,
            "overall_accuracy_pct": float(vals.mean()),
            "note": "Fixed n=1024 table run. Uncertainty is SEM over unordered Tiny10 model pairs.",
        })

    summary_df = pd.DataFrame(rows).sort_values(["family", "method", "K"]).reset_index(drop=True)
    summary_df.to_csv(LOCAL_SUMMARY_PATH, index=False)
    return pair_df, summary_df

if LOCAL_PAIR_PATH.exists():
    local_pair_df = pd.read_csv(LOCAL_PAIR_PATH)
else:
    local_pair_df = pd.DataFrame(columns=[
        "method", "family", "regime", "K", "model_i", "model_j", "pair_accuracy_pct"
    ])

LAYER_PAIRS = [(i, j) for i in range(len(LAYER_NAMES)) for j in range(len(LAYER_NAMES))]

# =========================================================
# Main local evaluation
# =========================================================

for family in families_sras.keys():
    print(f"\n================ FAMILY: {family} ================")

    for model_i, model_j in tqdm(PAIR_LIST, desc=f"{family}: model pairs"):
        if _local_pair_complete(local_pair_df, family, model_i, model_j):
            continue

        # Drop partial rows for this pair.
        if len(local_pair_df):
            mask = (
                (local_pair_df["family"].astype(str) == family) &
                (local_pair_df["model_i"].astype(int) == int(model_i)) &
                (local_pair_df["model_j"].astype(int) == int(model_j))
            )
            local_pair_df = local_pair_df.loc[~mask].copy()

        t0 = time.time()

        dist_maps = {}

        # --------------------------
        # S-RAS full and shape
        # --------------------------
        for method in ["sras_full", "sras_shape"]:
            for k in K_SRAS_ALL:
                dist_maps[(method, k)] = np.zeros((len(LAYER_NAMES), len(LAYER_NAMES)), dtype=np.float32)

        mean_i = {
            layer: np.load(_sras_mean_path(family, model_i, layer)).astype(np.float32)
            for layer in LAYER_NAMES
        }
        mean_j = {
            layer: np.load(_sras_mean_path(family, model_j, layer)).astype(np.float32)
            for layer in LAYER_NAMES
        }

        for a, layer_a in enumerate(LAYER_NAMES):
            for b, layer_b in enumerate(LAYER_NAMES):
                GA = mean_i[layer_a]
                GB = mean_j[layer_b]
                for k in K_SRAS_ALL:
                    dist_maps[("sras_full", k)][a, b] = _sras_distance(GA, GB, k, shape_only=False)
                    dist_maps[("sras_shape", k)][a, b] = _sras_distance(GA, GB, k, shape_only=True)

        # --------------------------
        # Pointwise controls
        # --------------------------
        for method in ["pwairm", "msa"]:
            for k in K_POINTWISE_ALL:
                dist_maps[(method, k)] = np.zeros((len(LAYER_NAMES), len(LAYER_NAMES)), dtype=np.float32)

        for k in K_POINTWISE_ALL:
            layer_batch = LAYERPAIR_BATCH_BY_K[k]

            for start in tqdm(
                range(0, len(LAYER_PAIRS), layer_batch),
                desc=f"{family} {model_i}-{model_j}: pointwise K={k}",
                leave=False,
            ):
                lp_block = LAYER_PAIRS[start:start + layer_batch]
                d_airm, d_msa = _pointwise_block_distances(family, model_i, model_j, lp_block, k)

                for idx, (a, b) in enumerate(lp_block):
                    dist_maps[("pwairm", k)][a, b] = d_airm[idx]
                    dist_maps[("msa", k)][a, b] = d_msa[idx]

        # --------------------------
        # Convert all maps to accuracy rows
        # --------------------------
        new_rows = []

        for method in ["sras_full", "sras_shape"]:
            for k in K_SRAS_ALL:
                sim = np.exp(-dist_maps[(method, k)]).astype(np.float32)
                acc = _pair_accuracy_from_sim_map(sim)
                new_rows.append({
                    "method": method,
                    "family": family,
                    "regime": REGIME_NAME,
                    "K": int(k),
                    "model_i": int(model_i),
                    "model_j": int(model_j),
                    "pair_accuracy_pct": float(acc),
                })

        for method in ["pwairm", "msa"]:
            for k in K_POINTWISE_ALL:
                sim = np.exp(-dist_maps[(method, k)]).astype(np.float32)
                acc = _pair_accuracy_from_sim_map(sim)
                new_rows.append({
                    "method": method,
                    "family": family,
                    "regime": REGIME_NAME,
                    "K": int(k),
                    "model_i": int(model_i),
                    "model_j": int(model_j),
                    "pair_accuracy_pct": float(acc),
                })

        local_pair_df = pd.concat([local_pair_df, pd.DataFrame(new_rows)], ignore_index=True, sort=False)
        local_pair_df, local_summary_df = _save_local(local_pair_df)

        print(
            f"Finished {family} pair {model_i}-{model_j} "
            f"in {(time.time() - t0)/60:.1f} min. Rows={len(local_pair_df)}"
        )

        del dist_maps, mean_i, mean_j
        gc.collect()
        try:
            jax.clear_caches()
        except Exception:
            pass

local_pair_df, local_summary_df = _save_local(local_pair_df)

print("Saved:")
print(" ", LOCAL_PAIR_PATH)
print(" ", LOCAL_SUMMARY_PATH)

display(local_summary_df)